# 02 — Inspect DICOMs (series-level grouping)

Reads downloaded DICOM headers from `data/raw/`, extracts useful metadata,
and groups by `SeriesInstanceUID` so each row is one imaging series.

Outputs:
- `outputs/dicom_file_metadata.csv` — per-file rows (contains PatientID, gitignored).
- `outputs/sample_series_summary.csv` — per-series rows (safe to commit, no PatientID).

In [ ]:
from pathlib import Path
import os

import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError

from pathlib import Path

# Works whether the notebook is launched from the repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
INPUT_DIR = REPO_ROOT / "data" / "raw"
OUT_DIR   = REPO_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PER_FILE_CSV    = OUT_DIR / "dicom_file_metadata.csv"
PER_SERIES_CSV  = OUT_DIR / "sample_series_summary.csv"

print("Input :", INPUT_DIR)
print("Out 1 :", PER_FILE_CSV)
print("Out 2 :", PER_SERIES_CSV)

## Recursively find candidate DICOM files

We don't filter by extension because Gen3 downloads sometimes drop the `.dcm` suffix.

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"{INPUT_DIR} does not exist. Run notebook 01 first to download a tiny sample."
    )

candidate_files = [p for p in INPUT_DIR.rglob("*") if p.is_file()]
print(f"Found {len(candidate_files)} candidate files under {INPUT_DIR}")
for p in candidate_files[:5]:
    print("  ", p)

## Read DICOM headers (no pixel data)

- `stop_before_pixels=True` keeps memory low.
- `force=True` tolerates files with missing/odd preambles.
- Unreadable files are skipped with a warning instead of crashing the run.

In [ ]:
HEADER_FIELDS = [
    "PatientID",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SOPInstanceUID",
    "Modality",
    "BodyPartExamined",
    "StudyDescription",
    "SeriesDescription",
    "ViewPosition",
    "ImageType",
    "Rows",
    "Columns",
    "Manufacturer",
]

def safe_get(ds, name):
    v = getattr(ds, name, None)
    if v is None:
        return ""
    if hasattr(v, "__iter__") and not isinstance(v, (str, bytes)):
        try:
            return "\\".join(str(x) for x in v)
        except Exception:
            return str(v)
    return str(v)

rows = []
skipped = 0
for p in candidate_files:
    try:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True)
    except (InvalidDicomError, OSError, Exception) as exc:
        print(f"  skip {p.name}: {exc}")
        skipped += 1
        continue
    row = {"path": str(p)}
    for f in HEADER_FIELDS:
        row[f] = safe_get(ds, f)
    rows.append(row)

df = pd.DataFrame(rows)
print(f"Parsed {len(df)} DICOM headers ({skipped} skipped).")
df.head()

## Save per-file metadata (gitignored)

PatientID is included here. This CSV is in the `.gitignore`.

In [ ]:
df.to_csv(PER_FILE_CSV, index=False)
print("Wrote", PER_FILE_CSV)

## Group by SeriesInstanceUID

Each row = one imaging series (per George Shih: classify at the series level).

In [ ]:
def first_nonempty(s):
    for v in s:
        if v not in ("", None):
            return v
    return ""

if df.empty or "SeriesInstanceUID" not in df.columns:
    series_df = pd.DataFrame(columns=[
        "SeriesInstanceUID", "StudyInstanceUID", "Modality",
        "BodyPartExamined", "StudyDescription", "SeriesDescription",
        "ViewPosition", "number_of_files", "example_path",
    ])
else:
    series_df = (
        df.groupby("SeriesInstanceUID", dropna=False)
          .agg(
              StudyInstanceUID  = ("StudyInstanceUID",  first_nonempty),
              Modality          = ("Modality",          first_nonempty),
              BodyPartExamined  = ("BodyPartExamined",  first_nonempty),
              StudyDescription  = ("StudyDescription",  first_nonempty),
              SeriesDescription = ("SeriesDescription", first_nonempty),
              ViewPosition      = ("ViewPosition",      first_nonempty),
              number_of_files   = ("path",              "count"),
              example_path      = ("path",              "first"),
          )
          .reset_index()
    )

series_df

In [ ]:
series_df.to_csv(PER_SERIES_CSV, index=False)
print(f"Wrote {len(series_df)} series rows -> {PER_SERIES_CSV}")

### Done.

If `series_df` is empty, double-check that notebook 01 actually wrote files
under `data/raw/`. If it shows rows but `Modality` / `BodyPartExamined` are
blank, the DICOMs themselves are missing those tags — that's expected for
real-world data and is exactly why the future classifier needs to look at
pixels too, not just headers.